# LAB 4 — Miniprojeto Blue Blood: o arco completo no sub-cenário do grupo

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 4 — Engenharia e Ciência de Dados + apresentação dos grupos (12 de novembro de 2026)
**Duração** manhã, das 08:00 às 12:00, em grupo
**Entrega** este notebook preenchido e a apresentação de 12 min à tarde; versão corrigida até **27/11/2026**

> **Para quem é este notebook:** gestores que decidem. Metade vem pronta e explicada; a outra metade o grupo completa nas células marcadas com **`TODO`**. Você não programa do zero: escolhe, preenche entre aspas, executa e interpreta.

## Como usar

1. Importe em **Workspace → (seu usuário) → Import → File → `lab-4.ipynb`** e conecte ao compute *Serverless*.
2. Execute as células **na ordem**, de cima para baixo. Células marcadas **🟢 PRONTA** só se executam. Células marcadas **🟡 TODO** o grupo preenche antes de executar.
3. A **Chefe de Inteligência** digita; a **Corregedora** confere a governança (Etapa 3); o **Comandante de Área** e a **Comunicação Social** preenchem a decisão e a nota; a **Comandante-Geral** fecha a decisão e apresenta.
4. A última célula confere o que falta, critério por critério da avaliação.

| Etapa do arco | Horário | Partes deste notebook |
| --- | --- | --- |
| 1 e 2. Pressão por decisão e critérios | 08:00–08:30 | Parte 1 |
| 3. Coleta e armazenamento | 08:30–09:45 | Partes 2 e 3 (Bronze, governança) |
| 4. Processamento e visualização | 10:00–11:15 | Partes 4 a 7 (Silver, Gold, painel, NLP, notícias) |
| 5. Decisão com critérios e risco | 11:15–12:00 | Partes 8 a 10 (decisão, política pública, catálogo, checagem) |

**Regras do caso.** A Blue Blood é fictícia; os dados de criminalidade são reais e públicos (ISP-RJ). A base do ISP Microdados que o grupo baixar **fica só no seu workspace**: não vai para o repositório, não vai por e-mail, não é repassada (Portaria ISP 127/2022).

In [0]:
# 🟢 PRONTA — preparação do ambiente. Só execute.
import io, re, json, datetime as dt
import numpy as np
import pandas as pd

try:
    spark  # existe no Databricks
    TEM_SPARK = True
except NameError:
    TEM_SPARK = False

TABELAS = {}      # nome -> DataFrame pandas (cópia local de cada tabela do pipeline)
DESCRICOES = {}   # nome -> descrição (vai para o catálogo)

def mostrar(df, n=20):
    try:
        display(df.head(n) if hasattr(df, "head") else df)
    except NameError:
        print(df.head(n).to_string() if hasattr(df, "head") else df)

def salvar(nome, df, descricao):
    """Guarda a tabela localmente e, no Databricks, como tabela Delta no schema do grupo."""
    TABELAS[nome] = df
    DESCRICOES[nome] = descricao
    if TEM_SPARK:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.{nome}")
        spark.sql(f"COMMENT ON TABLE {SCHEMA}.{nome} IS '{descricao.replace(chr(39), '')}'")
    print(f"Tabela {nome}: {len(df):,} linhas" + (f" → {SCHEMA}.{nome} (Delta)" if TEM_SPARK else " (memória local)"))

print("Databricks com Spark" if TEM_SPARK else "Sem Spark: modo de teste local com pandas")

---
## Parte 1 — Pressão por decisão e critérios (08:00–08:30)

Retomem o ensaio da aula 3 e o lab 1. Esta é a primeira célula que o professor lê: **que decisão o grupo vai sustentar?**

In [0]:
# 🟡 TODO — identificação do grupo e a decisão em jogo (copie do lab 1 e do ensaio da aula 3).

GRUPO = 0  # TODO: número do grupo, de 1 a 8 (sem aspas)

DECISAO = {
    "manchete_ou_pressao": "TODO",   # a notícia ou demanda que pressiona o comando
    "bairro_ou_cisp": "TODO",        # ex.: "CISP 12"
    "delito": "TODO",                # ex.: "roubo de rua"
    "decisao_em_jogo": "TODO",       # o que muda se o comando agir?
    "quem_decide": "TODO",
    "prazo": "TODO",
    "custo_de_errar": "TODO",
}

SCHEMA = f"workspace.blue_blood_g{GRUPO}"   # onde as tabelas Delta do grupo ficam no Databricks

---
## Parte 2 — Acesso com governança: ISP Microdados (08:30–09:00)

Cada integrante baixa a base **com as próprias credenciais**:

1. Acesse <https://ispconecta.rj.gov.br/microdados/> e preencha o formulário de identificação.
2. Escolha os filtros do sub-cenário: tipo de contagem (casos ou vítimas), delito, recorte territorial (CISP, AISP, RISP, bairro ou município) e período.
3. Gere e baixe a base. **Anote a data e os filtros.**
4. No Databricks: **Catalog → workspace → default → Create → Volume** chamado `isp_microdados`; abra o volume e faça **Upload** do arquivo.

> **Portaria ISP 127/2022.** Os microdados são fornecidos mediante Termo de Responsabilidade: uso restrito a estudos próprios voltados a políticas públicas, **proibido comercializar ou repassar a terceiros**, e inconsistências devem ser comunicadas ao ISP. O professor não baixa nem distribui a base.

**Evidência obrigatória (critério Governança, 20%):** data do download, filtros usados e compromisso com o Termo. Sem os três, a decisão não vale.

In [0]:
# 🟡 TODO — registro do download (a Corregedora confere).

REGISTRO_DOWNLOAD = {
    "data_download": "TODO",            # ex.: "2026-11-12"
    "responsavel": "TODO",              # quem baixou, com a própria credencial
    "filtros": {
        "tipo_contagem": "TODO",        # "casos" ou "vítimas"
        "delitos": "TODO",              # ex.: "roubo a transeunte; roubo de celular"
        "recorte_territorial": "TODO",  # ex.: "CISP 12 e CISP 13"
        "periodo": "TODO",              # ex.: "2024-01 a 2026-09"
    },
    "arquivo": "TODO",                  # nome do arquivo baixado
    "termo_de_responsabilidade": False, # TODO: True quando o grupo assumir o compromisso abaixo
    "compromisso": "Usaremos a base só neste estudo, sem comercializar nem repassar a terceiros, "
                   "e comunicaremos ao ISP qualquer inconsistência (Portaria ISP 127/2022).",
}

---
## Parte 3 — Bronze: o dado como chegou (09:00–09:45)

Escolham a fonte. Se o formulário do ISP Microdados falhar no dia, usem o **plano B** (séries públicas do ISPDados). Sem internet, o notebook usa uma **amostra real embutida** do ISPDados (CISPs 12 e 13, jan/2024 a set/2026), só para o pipeline rodar.

Se usarem o ISP Microdados, digam quais colunas do arquivo correspondem a CISP, ano, mês e delito. Rodem primeiro com `MAPA_COLUNAS` como está: a célula seguinte mostra as colunas do arquivo.

In [0]:
# 🟡 TODO — fonte e mapeamento de colunas.

FONTE = "microdados"   # TODO: "microdados" (preferido), "ispdados" (plano B) ou "exemplo" (amostra embutida)

CAMINHO_MICRODADOS = "/Volumes/workspace/default/isp_microdados/TODO.csv"  # TODO: caminho do arquivo no volume

# Só para FONTE = "microdados": nome da coluna do arquivo para cada campo.
MAPA_COLUNAS = {
    "cisp": "TODO",
    "ano": "TODO",
    "mes": "TODO",
    "delito": "TODO",
    "contagem": None,   # nome da coluna com a quantidade; None se cada linha for um caso ou vítima
}

In [0]:
# 🟢 PRONTA — ingestão na Bronze (sem alterar o conteúdo) e metadados de origem. Só execute.

URL_ISPDADOS = "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv"
URL_POPULACAO = "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv"

AMOSTRA_ISPDADOS = """cisp,aisp,risp,munic,ano,mes,fase,roubo_rua,roubo_transeunte,roubo_celular,total_roubos
12,19,1,Rio de Janeiro,2024,1,3,61,29,31,68
13,19,1,Rio de Janeiro,2024,1,3,24,19,4,29
12,19,1,Rio de Janeiro,2024,2,3,35,20,14,50
13,19,1,Rio de Janeiro,2024,2,3,22,16,6,29
12,19,1,Rio de Janeiro,2024,3,3,37,27,10,48
13,19,1,Rio de Janeiro,2024,3,3,12,6,5,21
12,19,1,Rio de Janeiro,2024,4,3,24,15,9,30
13,19,1,Rio de Janeiro,2024,4,3,22,15,4,35
12,19,1,Rio de Janeiro,2024,5,3,93,48,43,109
13,19,1,Rio de Janeiro,2024,5,3,34,12,22,38
12,19,1,Rio de Janeiro,2024,6,3,39,19,18,51
13,19,1,Rio de Janeiro,2024,6,3,17,9,8,24
12,19,1,Rio de Janeiro,2024,7,3,31,20,11,39
13,19,1,Rio de Janeiro,2024,7,3,10,4,4,18
12,19,1,Rio de Janeiro,2024,8,3,40,27,13,50
13,19,1,Rio de Janeiro,2024,8,3,15,9,6,18
12,19,1,Rio de Janeiro,2024,9,3,37,21,13,44
13,19,1,Rio de Janeiro,2024,9,3,22,15,7,29
12,19,1,Rio de Janeiro,2024,10,3,26,18,8,46
13,19,1,Rio de Janeiro,2024,10,3,10,6,4,15
12,19,1,Rio de Janeiro,2024,11,3,25,17,7,38
13,19,1,Rio de Janeiro,2024,11,3,14,8,5,19
12,19,1,Rio de Janeiro,2024,12,3,30,14,13,42
13,19,1,Rio de Janeiro,2024,12,3,12,9,3,19
12,19,1,Rio de Janeiro,2025,1,3,57,36,18,65
13,19,1,Rio de Janeiro,2025,1,3,22,14,8,27
12,19,1,Rio de Janeiro,2025,2,3,17,9,7,26
13,19,1,Rio de Janeiro,2025,2,3,18,9,9,23
12,19,1,Rio de Janeiro,2025,3,3,32,21,11,35
13,19,1,Rio de Janeiro,2025,3,3,25,21,4,34
12,19,1,Rio de Janeiro,2025,4,3,20,12,7,26
13,19,1,Rio de Janeiro,2025,4,3,22,16,6,30
12,19,1,Rio de Janeiro,2025,5,3,64,24,39,79
13,19,1,Rio de Janeiro,2025,5,3,18,12,5,27
12,19,1,Rio de Janeiro,2025,6,3,20,12,6,33
13,19,1,Rio de Janeiro,2025,6,3,14,7,7,22
12,19,1,Rio de Janeiro,2025,7,3,28,16,12,47
13,19,1,Rio de Janeiro,2025,7,3,12,10,2,21
12,19,1,Rio de Janeiro,2025,8,3,34,12,19,49
13,19,1,Rio de Janeiro,2025,8,3,14,6,8,23
12,19,1,Rio de Janeiro,2025,9,3,22,11,11,30
13,19,1,Rio de Janeiro,2025,9,3,14,10,4,20
12,19,1,Rio de Janeiro,2025,10,3,22,15,4,31
13,19,1,Rio de Janeiro,2025,10,3,18,14,4,24
12,19,1,Rio de Janeiro,2025,11,3,25,13,11,34
13,19,1,Rio de Janeiro,2025,11,3,21,9,10,28
12,19,1,Rio de Janeiro,2025,12,3,24,10,12,28
13,19,1,Rio de Janeiro,2025,12,3,25,15,9,32
12,19,1,Rio de Janeiro,2026,1,3,36,19,17,47
13,19,1,Rio de Janeiro,2026,1,3,26,13,11,29
12,19,1,Rio de Janeiro,2026,2,3,32,16,15,36
13,19,1,Rio de Janeiro,2026,2,3,18,8,10,24
12,19,1,Rio de Janeiro,2026,3,3,25,16,9,33
13,19,1,Rio de Janeiro,2026,3,3,18,13,5,22
12,19,1,Rio de Janeiro,2026,4,3,25,14,9,29
13,19,1,Rio de Janeiro,2026,4,3,9,3,6,14
12,19,1,Rio de Janeiro,2026,5,3,70,35,34,78
13,19,1,Rio de Janeiro,2026,5,3,22,14,7,24
12,19,1,Rio de Janeiro,2026,6,3,26,8,18,31
13,19,1,Rio de Janeiro,2026,6,3,16,7,9,20
12,19,1,Rio de Janeiro,2026,7,2,20,10,10,28
13,19,1,Rio de Janeiro,2026,7,2,10,6,4,12
12,19,1,Rio de Janeiro,2026,8,2,21,14,5,33
13,19,1,Rio de Janeiro,2026,8,2,10,4,6,15
12,19,1,Rio de Janeiro,2026,9,2,20,10,9,27
13,19,1,Rio de Janeiro,2026,9,2,10,6,4,14"""
AMOSTRA_POPULACAO = """circ,mes,ano,pop_circ
12,7,2022,98287.55867
13,7,2022,74436.86263"""

def ler_csv(origem, **kw):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(origem, sep=None, engine="python", encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não consegui ler {origem}")

fonte_usada, bronze = None, None
if FONTE == "microdados":
    try:
        bronze = ler_csv(CAMINHO_MICRODADOS)
        fonte_usada = "ISP Microdados"
    except Exception as e:
        print(f"ISP Microdados não lido ({type(e).__name__}). Indo para o plano B.")
if bronze is None and FONTE in ("microdados", "ispdados"):
    try:
        bronze = pd.read_csv(URL_ISPDADOS, sep=";", encoding="latin-1")
        fonte_usada = "ISPDados (séries públicas)"
    except Exception as e:
        print(f"ISPDados indisponível ({type(e).__name__}). Usando a amostra embutida.")
if bronze is None:
    bronze = pd.read_csv(io.StringIO(AMOSTRA_ISPDADOS))
    fonte_usada = "Amostra embutida do ISPDados (CISPs 12 e 13)"

bronze["_fonte"] = fonte_usada
bronze["_ingerido_em"] = dt.datetime.now().isoformat(timespec="seconds")
salvar("bronze_isp", bronze, f"Bronze: {fonte_usada}, como chegou, sem alteração")
print("Colunas do arquivo:", [c for c in bronze.columns if not c.startswith("_")][:40])

---
## Parte 4 — Silver: limpo, tipado e checado (10:00–10:25)

A Silver põe tudo no mesmo formato: **uma linha por CISP, mês e delito**. Depois vêm as checagens de qualidade (critério Governança).

**Decisão do grupo:** os meses mais recentes do ISPDados vêm com `fase = 2` (dados ainda não consolidados). Manter ou excluir?

In [0]:
# 🟡 TODO — regra para meses preliminares.

REGRA_PRELIMINAR = "manter"   # TODO: "manter" (marcando como preliminar) ou "excluir"

In [0]:
# 🟢 PRONTA — Silver em formato longo + checagens de qualidade. Só execute.

ID = ["cisp", "ano", "mes"]
if fonte_usada == "ISP Microdados":
    faltam = [k for k in ("cisp", "ano", "mes", "delito") if MAPA_COLUNAS[k] in (None, "TODO")]
    if faltam:
        raise ValueError(f"Preencha MAPA_COLUNAS para: {faltam}. Colunas disponíveis: {list(bronze.columns)}")
    m = {MAPA_COLUNAS[k]: k for k in ("cisp", "ano", "mes", "delito")}
    base = bronze.rename(columns=m)
    if MAPA_COLUNAS["contagem"]:
        base = base.rename(columns={MAPA_COLUNAS["contagem"]: "ocorrencias"})
    else:
        base["ocorrencias"] = 1
    silver = base.groupby(ID + ["delito"], as_index=False)["ocorrencias"].sum()
    silver["fase"] = 3
else:
    delitos = [c for c in bronze.columns if c.startswith(("roubo", "furto", "total_", "hom_", "letalidade", "estupro", "estelionato"))]
    silver = bronze.melt(id_vars=ID + ["fase"], value_vars=delitos, var_name="delito", value_name="ocorrencias")

silver["delito"] = silver["delito"].astype(str).str.strip().str.lower()
for c in ID:
    silver[c] = pd.to_numeric(silver[c], errors="coerce")
silver["ocorrencias"] = pd.to_numeric(silver["ocorrencias"], errors="coerce")

qualidade = pd.DataFrame([
    {"checagem": "Completude: linhas sem CISP, ano ou mês", "dimensao": "Completude",
     "problemas": int(silver[ID].isna().any(axis=1).sum())},
    {"checagem": "Ocorrências vazias", "dimensao": "Completude", "problemas": int(silver["ocorrencias"].isna().sum())},
    {"checagem": "Ocorrências negativas", "dimensao": "Validade", "problemas": int((silver["ocorrencias"] < 0).sum())},
    {"checagem": "Mês fora de 1 a 12", "dimensao": "Validade", "problemas": int((~silver["mes"].between(1, 12)).sum())},
    {"checagem": "Linhas duplicadas (CISP, mês, delito)", "dimensao": "Unicidade",
     "problemas": int(silver.duplicated(ID + ["delito"]).sum())},
    {"checagem": "Linhas em fase preliminar (fase 2)", "dimensao": "Atualidade",
     "problemas": int((silver.get("fase", 3) == 2).sum())},
])
silver = silver.dropna(subset=ID + ["ocorrencias"])
silver = silver[(silver["ocorrencias"] >= 0) & silver["mes"].between(1, 12)].drop_duplicates(ID + ["delito"])
if REGRA_PRELIMINAR == "excluir":
    silver = silver[silver["fase"] != 2]
silver = silver.astype({"cisp": int, "ano": int, "mes": int})
silver["data"] = pd.to_datetime(dict(year=silver["ano"], month=silver["mes"], day=1))

mostrar(qualidade)
salvar("silver_isp", silver, "Silver: ocorrências por CISP, mês e delito, tipadas e checadas")
salvar("qualidade_silver", qualidade, "Scorecard de qualidade da Silver (DMBOK: completude, validade, unicidade, atualidade)")
print("Delitos disponíveis:", sorted(silver["delito"].unique())[:30])
print("CISPs disponíveis:", sorted(silver["cisp"].unique())[:40], "...")

---
## Parte 5 — Gold: os indicadores do sub-cenário (10:25–10:45)

A Gold responde à pergunta do grupo. Dois indicadores:

- **Taxa por 100 mil habitantes**: compara áreas de tamanhos diferentes (número absoluto engana).
- **Variação da janela recente contra a mesma janela do ano anterior**: separa alta real de sazonalidade.

> A série de população do ISP vai até jul/2022; para meses posteriores usamos a última população disponível. Registrem essa limitação na apresentação.

In [0]:
# 🟡 TODO — o recorte do grupo.

CISP_FOCO = 12               # TODO: CISP do grupo (número, sem aspas)
DELITO_FOCO = "roubo_rua"    # TODO: um dos delitos listados na Silver
JANELA_MESES = 3             # TODO: tamanho da janela de comparação (a manchete fala em 3 meses)
VIZINHAS = "mesma_aisp"      # TODO: "mesma_aisp" ou uma lista de CISPs, ex.: [12, 13, 14]

In [0]:
# 🟢 PRONTA — população, taxa por 100 mil e variação. Só execute.

try:
    pop = pd.read_csv(URL_POPULACAO, sep=";", decimal=",")
except Exception:
    pop = pd.read_csv(io.StringIO(AMOSTRA_POPULACAO))
pop = pop.rename(columns={"circ": "cisp"})
pop_ultima = pop.sort_values(["ano", "mes"]).groupby("cisp", as_index=False).last()[["cisp", "pop_circ"]]

if VIZINHAS == "mesma_aisp" and "aisp" in bronze.columns:
    aisp = bronze.loc[bronze["cisp"] == CISP_FOCO, "aisp"].iloc[-1]
    area = sorted(bronze.loc[bronze["aisp"] == aisp, "cisp"].unique())
elif isinstance(VIZINHAS, list):
    area = sorted(set(VIZINHAS) | {CISP_FOCO})
else:
    area = [CISP_FOCO]

g = silver[(silver["delito"] == DELITO_FOCO) & silver["cisp"].isin(area)].merge(pop_ultima, on="cisp", how="left")
g["taxa_100mil"] = (g["ocorrencias"] / g["pop_circ"] * 1e5).round(2)
gold_serie = g.sort_values(["cisp", "data"])[["cisp", "data", "ano", "mes", "delito", "ocorrencias", "pop_circ", "taxa_100mil", "fase"]]

ultimo = gold_serie["data"].max()
inicio = ultimo - pd.DateOffset(months=JANELA_MESES - 1)
linhas = []
for c, d in gold_serie.groupby("cisp"):
    rec = d[(d["data"] >= inicio) & (d["data"] <= ultimo)]["ocorrencias"].sum()
    ant = d[(d["data"] >= inicio - pd.DateOffset(years=1)) & (d["data"] <= ultimo - pd.DateOffset(years=1))]["ocorrencias"].sum()
    linhas.append({"cisp": c, "janela": f"{inicio:%Y-%m} a {ultimo:%Y-%m}", "ocorrencias_janela": int(rec),
                   "mesma_janela_ano_anterior": int(ant),
                   "variacao_pct": round((rec - ant) / ant * 100, 1) if ant else None,
                   "taxa_100mil_janela": round(rec / d["pop_circ"].iloc[0] * 1e5, 1) if d["pop_circ"].notna().any() else None})
gold_variacao = pd.DataFrame(linhas)

salvar("gold_serie", gold_serie, f"Gold: série mensal de {DELITO_FOCO} nas CISPs {area}, com taxa por 100 mil")
salvar("gold_variacao", gold_variacao, f"Gold: variação de {JANELA_MESES} meses contra o mesmo período do ano anterior")
mostrar(gold_variacao)
VARIACAO_FOCO = gold_variacao.set_index("cisp").loc[CISP_FOCO, "variacao_pct"]
print(f"CISP {CISP_FOCO}, {DELITO_FOCO}: variação de {VARIACAO_FOCO}% na janela {inicio:%Y-%m} a {ultimo:%Y-%m}")

---
## Parte 6 — Painel (10:45–10:55)

Dois gráficos que a Comandante-Geral precisa ler em 30 segundos: a série em **número absoluto** e em **taxa por 100 mil**, com as CISPs vizinhas. Perguntas para o grupo: a conclusão muda entre os dois gráficos? A alta é só na CISP do grupo ou na área toda?

In [0]:
# 🟢 PRONTA — painel do sub-cenário. Só execute.
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for c, d in gold_serie.groupby("cisp"):
    destaque = dict(lw=2.6, color="#0b2a4a") if c == CISP_FOCO else dict(lw=1.2, alpha=0.6)
    eixos[0].plot(d["data"], d["ocorrencias"], label=f"CISP {c}", **destaque)
    eixos[1].plot(d["data"], d["taxa_100mil"], label=f"CISP {c}", **destaque)
for ax, titulo in zip(eixos, ["Ocorrências por mês", "Taxa por 100 mil habitantes"]):
    ax.set_title(f"{titulo} — {DELITO_FOCO}")
    ax.axvspan(inicio, ultimo, color="#1d6fb8", alpha=0.08)
    ax.grid(alpha=0.3)
eixos[0].legend()
fig.text(0.5, -0.02, f"Fonte: {fonte_usada}. Faixa azul: janela recente de {JANELA_MESES} meses.", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## Parte 7 — NLP sobre narrativas de boletins (10:55–11:05)

O histórico do BO, o dado mais rico, **não é público** (LAI art. 31, LGPD). Usamos o corpus **KDMiLe 2026**: 996 narrativas reais de roubo da SSP-SP, descaracterizadas (nomes fictícios), com padrão-ouro de dados pessoais. Licença CC BY 4.0 — cite: Almeida, Botega e Lima, *Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese*, KDMiLe/SBC, 2026.

Duas tarefas: **classificar o modo de agir** por palavras-chave e **conferir a anonimização** (quantos telefones e placas um detector simples encontra, comparado com o padrão-ouro).

In [0]:
# 🟢 PRONTA — carga do corpus KDMiLe na Bronze. Só execute.

URL_KDMILE = "https://github.com/murilogoes/bo-anonymizer-ptbr/raw/main/data/bos_anonimizacao_CAP_descaracterizado.xlsx"

NARRATIVAS_SINTETICAS = pd.DataFrame({  # fallback sem internet: narrativas SINTÉTICAS, nomes fictícios
    "Seq": [1, 2, 3, 4, 5],
    "Texto Descaracterizado": [
        "[SINTÉTICO] A vítima caminhava pela calçada quando dois indivíduos em uma motocicleta anunciaram o assalto com arma de fogo e levaram seu celular.",
        "[SINTÉTICO] O autor, portando uma faca, abordou a vítima no ponto de ônibus e subtraiu a bolsa e o aparelho celular. Contato da vítima: (21) 99999-0000.",
        "[SINTÉTICO] Três homens armados renderam o motorista e fugiram com o veículo de placa ABC1D23.",
        "[SINTÉTICO] Dentro do coletivo, o indivíduo simulou estar armado e recolheu celulares dos passageiros.",
        "[SINTÉTICO] O comerciante relatou que o autor entrou na loja com um simulacro de pistola e levou o dinheiro do caixa.",
    ],
    "Telefone (s)": [None, "(21) 99999-0000", None, None, None],
    "Emplacamento (s)": [None, None, "ABC1D23", None, None],
})
try:
    narrativas = pd.read_excel(URL_KDMILE)
    fonte_nlp = "KDMiLe 2026 (CC BY 4.0)"
except Exception as e:
    narrativas = NARRATIVAS_SINTETICAS.copy()
    fonte_nlp = "Narrativas sintéticas de exemplo (sem internet)"
narrativas = narrativas.rename(columns={"Texto Descaracterizado": "texto"})
salvar("bronze_narrativas", narrativas[["Seq", "texto"]], f"Bronze: narrativas de BO — {fonte_nlp}")
print(f"{len(narrativas)} narrativas carregadas de: {fonte_nlp}")

In [0]:
# 🟡 TODO — categorias do modo de agir. Ajustem ou acrescentem UMA categoria ligada ao delito do grupo.

CATEGORIAS = {
    "arma de fogo": ["arma de fogo", "revólver", "revolver", "pistola", "armado", "armados"],
    "arma branca": ["faca", "canivete", "estilete"],
    "simulacro ou ameaça": ["simulacro", "simulou", "fingiu estar armado", "ameaç"],
    "motocicleta": ["motocicleta", "moto ", "garupa"],
    "celular levado": ["celular", "aparelho telefônico", "smartphone"],
    # "TODO: nova categoria": ["palavra 1", "palavra 2"],
}

In [0]:
# 🟢 PRONTA — classificação por palavras-chave e checagem de anonimização. Só execute.

texto = narrativas["texto"].fillna("").str.lower()
for cat, termos in CATEGORIAS.items():
    narrativas[cat] = texto.apply(lambda t: any(term in t for term in termos))
modo = pd.DataFrame({"categoria": list(CATEGORIAS),
                     "narrativas": [int(narrativas[c].sum()) for c in CATEGORIAS]})
modo["percentual"] = (modo["narrativas"] / len(narrativas) * 100).round(1)
salvar("gold_modo_de_agir", modo.sort_values("narrativas", ascending=False), "Gold: modo de agir nas narrativas, por palavras-chave")
mostrar(modo.sort_values("narrativas", ascending=False))

RE_TELEFONE = re.compile(r"\(?\d{2}\)?[\s-]?\d{4,5}-?\d{4}")
RE_PLACA = re.compile(r"\b[A-Z]{3}-?\d[A-Z0-9]\d{2}\b")
def recall(coluna, padrao):
    com_ouro = narrativas[coluna].notna()
    achou = narrativas.loc[com_ouro, "texto"].fillna("").apply(lambda t: bool(padrao.search(t)))
    return int(com_ouro.sum()), int(achou.sum())
anon = []
for nome, col, pad in [("Telefone", "Telefone (s)", RE_TELEFONE), ("Placa", "Emplacamento (s)", RE_PLACA)]:
    if col in narrativas.columns:
        total, achou = recall(col, pad)
        anon.append({"dado_pessoal": nome, "narrativas_com_o_dado": total, "detectadas_pelo_padrao": achou,
                     "recall_pct": round(achou / total * 100, 1) if total else None})
anon = pd.DataFrame(anon)
mostrar(anon)
narrativas["texto_mascarado"] = narrativas["texto"].fillna("").str.replace(RE_TELEFONE, "[TELEFONE]", regex=True).str.replace(RE_PLACA, "[PLACA]", regex=True)
salvar("silver_narrativas_mascaradas", narrativas[["Seq", "texto_mascarado"]], "Silver: narrativas com telefones e placas mascarados")
print("Recall abaixo de 100% = dado pessoal vazaria. Nomes e endereços exigem modelos (aula 2), não só padrões.")

---
## Parte 7b — Checagem das notícias contra o ISP (11:05–11:15)

O Chefe de Comunicação Social traz as notícias (Google News). **Cada notícia precisa estar ligada a uma série do ISP**: CISP, delito, período e base de comparação (critério Evidência ISP, 20%). Para cada uma, digitem a variação que a notícia afirma; a célula seguinte compara com a Gold.

In [0]:
# 🟡 TODO — notícias a checar (uma por bloco; copiem o bloco para mais notícias).

NOTICIAS = [
    {
        "titulo": "TODO",
        "veiculo_e_data": "TODO",
        "url": "TODO",
        "cisp": CISP_FOCO,                  # série do ISP ligada à notícia
        "delito": DELITO_FOCO,
        "variacao_afirmada_pct": None,      # TODO: número que a notícia afirma, ex.: 40
        "periodo_afirmado": "TODO",         # ex.: "últimos três meses"
    },
]

TOLERANCIA_PP = 10   # TODO: diferença (em pontos percentuais) até a qual a notícia é considerada compatível

In [0]:
# 🟢 PRONTA — confronto de cada notícia com a série do ISP. Só execute.

linhas = []
for n in NOTICIAS:
    serie = gold_variacao.set_index("cisp")
    medido = serie.loc[n["cisp"], "variacao_pct"] if (n["delito"] == DELITO_FOCO and n["cisp"] in serie.index) else None
    if n["variacao_afirmada_pct"] is None or medido is None:
        veredito = "Sem checagem: falta a variação afirmada ou a série não está na Gold"
    elif abs(n["variacao_afirmada_pct"] - medido) <= TOLERANCIA_PP:
        veredito = "Compatível com o ISP"
    elif np.sign(n["variacao_afirmada_pct"]) != np.sign(medido):
        veredito = "Contraria o ISP (sinal oposto)"
    else:
        veredito = "Exagera ou minimiza o ISP"
    linhas.append({"noticia": n["titulo"], "serie_isp": f"CISP {n['cisp']} · {n['delito']} · {JANELA_MESES} meses vs ano anterior",
                   "afirmado_pct": n["variacao_afirmada_pct"], "medido_pct": medido, "veredito": veredito})
checagem_noticias = pd.DataFrame(linhas)
salvar("gold_checagem_noticias", checagem_noticias, "Gold: cada notícia confrontada com uma série específica do ISP")
mostrar(checagem_noticias)

**Segunda opinião com IA (opcional, 5 min).** A célula abaixo pede ao `ai_decide` do Databricks (o mesmo julgamento tipado da Parte 6 do lab 2) a probabilidade de cada notícia ser sustentada pela série do ISP, direto sobre a tabela Gold. O veredito que vale para o trabalho continua sendo o da célula acima; a IA é triagem. Se as duas discordarem, a notícia vai para revisão humana. Se o workspace não tiver `ai_decide` (Beta, só em algumas regiões), a célula avisa e segue.

In [ ]:
# 🟢 PRONTA — segunda opinião com ai_decide sobre a Gold de notícias. Só execute.
PERGUNTA_EVIDENCIA = json.dumps({"evidencia_isp": {
    "type": "noul",
    "instructions": "A variação afirmada pela notícia é sustentada pela variação medida na série do ISP, no mesmo recorte e período?",
    "criteria": {"true": "A variação medida pelo ISP confirma a afirmada, no mesmo recorte e período",
                 "false": "A variação medida contradiz a afirmada, ou falta a afirmada ou a medida"},
}}, ensure_ascii=False).replace("\\", "\\\\").replace("'", "\\'")

def regra(p):
    return "sustentada" if p >= 0.7 else ("revisão humana" if p >= 0.4 else "não sustentada")

try:
    ia = spark.sql(f"""
        SELECT noticia,
               variant_get(ai_decide(to_json(named_struct('noticia', noticia, 'serie_isp', serie_isp,
                                                          'afirmado_pct', afirmado_pct, 'medido_pct', medido_pct)),
                                     '{PERGUNTA_EVIDENCIA}'),
                           '$.response.answers.evidencia_isp.probability', 'double') AS p_sustentada_ia
        FROM {SCHEMA}.gold_checagem_noticias
    """).toPandas()
    segunda = checagem_noticias.merge(ia, on="noticia", how="left")
    segunda["regra_ia"] = segunda["p_sustentada_ia"].map(lambda p: regra(p) if pd.notna(p) else "sem resposta")
    segunda["concordam"] = (segunda["veredito"] == "Compatível com o ISP") == (segunda["regra_ia"] == "sustentada")
    mostrar(segunda[["noticia", "veredito", "p_sustentada_ia", "regra_ia", "concordam"]])
    print("Onde 'concordam' for False, a notícia vai para revisão humana antes da nota pública.")
except Exception as e:
    print(f"ai_decide indisponível ({type(e).__name__}); vale o veredito da célula anterior.")

---
## Parte 8 — Decisão com critérios, pesos e risco (11:15–11:35)

Tragam do lab 1 os critérios e os julgamentos par a par (escala de Saaty). O limiar de Evidência agora é testado contra o número da Gold.

In [0]:
# 🟡 TODO — critérios, julgamentos (do lab 1), limiar de evidência e alternativas.

CRITERIOS = ["Evidência", "Custo operacional", "Legitimidade"]   # TODO: os do grupo
JULGAMENTOS = {                                                    # TODO: os do grupo
    ("Evidência", "Custo operacional"): 3,
    ("Evidência", "Legitimidade"): 2,
    ("Legitimidade", "Custo operacional"): 2,
}
LIMIAR_EVIDENCIA_PCT = 15   # TODO: a partir de que alta (%) o comando age?

ALTERNATIVAS = {            # TODO: notas de 1 a 5 (em custo, 5 = menor custo)
    "Reforçar a CISP do grupo com efetivo da vizinha": {"Evidência": 3, "Custo operacional": 2, "Legitimidade": 4},
    "Manter o efetivo e publicar nota com os dados": {"Evidência": 3, "Custo operacional": 5, "Legitimidade": 3},
}

In [0]:
# 🟢 PRONTA — pesos AHP, consistência, limiar e pontuação. Só execute.

def pesos_ahp(criterios, julgamentos):
    n = len(criterios); idx = {c: i for i, c in enumerate(criterios)}; A = np.ones((n, n))
    for (a, b), v in julgamentos.items():
        A[idx[a], idx[b]] = v; A[idx[b], idx[a]] = 1 / v
    w = np.prod(A, axis=1) ** (1 / n); w = w / w.sum()
    lam = (A @ w / w).mean(); ia = {3: 0.58, 4: 0.90, 5: 1.12}
    return w, (((lam - n) / (n - 1)) / ia[n] if n >= 3 else 0.0)

w, CR = pesos_ahp(CRITERIOS, JULGAMENTOS)
PESOS = dict(zip(CRITERIOS, w.round(3)))
print("Pesos:", {k: f"{v:.0%}" for k, v in PESOS.items()}, f"| CR = {CR:.3f}", "(coerente)" if CR < 0.10 else "(REVER julgamentos)")
LIMIAR_ATINGIDO = VARIACAO_FOCO is not None and VARIACAO_FOCO >= LIMIAR_EVIDENCIA_PCT
print(f"Evidência: variação de {VARIACAO_FOCO}% contra limiar de {LIMIAR_EVIDENCIA_PCT}% → "
      + ("limiar ATINGIDO: os dados sustentam agir" if LIMIAR_ATINGIDO else "limiar NÃO atingido: os dados não sustentam a alta da manchete"))

ranking = pd.DataFrame([{"alternativa": a, **notas, "pontuacao": round(sum(notas[c] * PESOS[c] for c in CRITERIOS), 2)}
                        for a, notas in ALTERNATIVAS.items()]).sort_values("pontuacao", ascending=False)
salvar("gold_decisao", ranking, "Gold: alternativas pontuadas pelos critérios e pesos do grupo")
mostrar(ranking)

In [0]:
# 🟡 TODO — a decisão final, o risco de errar e a política pública.

DECISAO_FINAL = {
    "decisao": "TODO",                    # o que o comando decide
    "indicador_que_sustenta": "TODO",     # ex.: "roubo de rua na CISP 12: +18% no trimestre contra o ano anterior (ISP)"
    "risco_de_errar": {
        "o_que_pode_estar_errado": "TODO",   # ex.: "meses de 2026 ainda preliminares; população de 2022"
        "consequencia_se_errar": "TODO",
        "como_vamos_perceber": "TODO",       # que número, em quanto tempo
    },
    "nota_publica": "TODO",               # 2 frases com dado, período, fonte e risco (Comunicação Social)
}

POLITICA_PUBLICA = {
    "publico": "TODO",                    # quem é atendido
    "acao": "TODO",                       # o que será feito
    "responsavel": "TODO",
    "indicador_de_acompanhamento": "TODO",
    "meta_e_prazo": "TODO",               # ex.: "reduzir a taxa em 10% em 6 meses"
}

---
## Parte 9 — Catálogo das tabelas (11:35–11:45)

Toda tabela tem dono, origem e sensibilidade. Preencham o dono de cada uma (um papel do Alto Comando).

In [0]:
# 🟡 TODO — dono de cada tabela (um papel do Alto Comando).

DONOS = {nome: "TODO" for nome in TABELAS}
# Exemplo: DONOS["bronze_isp"] = "Corregedora e Governança de Dados"

In [0]:
# 🟢 PRONTA — catálogo do pipeline. Só execute.

SENSIBILIDADE = {"bronze_isp": "Restrita: base do ISP sob Termo de Responsabilidade; não repassar",
                 "bronze_narrativas": "Pública com atribuição (CC BY 4.0); narrativas descaracterizadas"}
catalogo = pd.DataFrame([{"tabela": n, "camada": n.split("_")[0], "descricao": DESCRICOES[n], "linhas": len(df),
                          "dono": DONOS.get(n, "TODO"), "sensibilidade": SENSIBILIDADE.get(n, "Interna: derivada e agregada")}
                         for n, df in TABELAS.items()])
mostrar(catalogo, n=50)
TABELAS["catalogo"] = catalogo
if TEM_SPARK:
    spark.createDataFrame(catalogo).write.mode("overwrite").saveAsTable(f"{SCHEMA}.catalogo")

---
## Parte 10 — Checagem final (11:45–12:00)

A célula confere o notebook pelos critérios da avaliação e monta o roteiro da apresentação de 12 min.

In [0]:
# 🟢 PRONTA — confere o que falta, por critério da avaliação. Só execute.

def preenchido(v):
    if v is None: return False
    if isinstance(v, str): return v.strip() not in ("", "TODO") and "TODO" not in v
    if isinstance(v, dict): return all(preenchido(x) for x in v.values())
    if isinstance(v, (list, tuple)): return len(v) > 0 and all(preenchido(x) for x in v)
    return True

r = REGISTRO_DOWNLOAD
itens = [
    ("Decisão acionável (25%)", "Decisão em jogo nomeada (Parte 1)", isinstance(GRUPO, int) and 1 <= GRUPO <= 8 and preenchido(DECISAO)),
    ("Decisão acionável (25%)", "Decisão final, indicador e risco de errar (Parte 8)", preenchido(DECISAO_FINAL)),
    ("Decisão acionável (25%)", "Julgamentos coerentes, CR < 0,10 (Parte 8)", CR < 0.10),
    ("Evidência ISP (20%)", "Ao menos uma notícia checada contra uma série", any(not l.startswith("Sem checagem") for l in checagem_noticias["veredito"])),
    ("Evidência ISP (20%)", "Toda notícia com título e fonte", all(preenchido([n["titulo"], n["veiculo_e_data"]]) for n in NOTICIAS)),
    ("Pipeline e IA (20%)", "Bronze, Silver e Gold criadas", all(t in TABELAS for t in ("bronze_isp", "silver_isp", "gold_serie", "gold_variacao"))),
    ("Pipeline e IA (20%)", "NLP sobre as narrativas (modo de agir e anonimização)", "gold_modo_de_agir" in TABELAS),
    ("Política pública (15%)", "Público, ação, responsável, indicador e meta (Parte 8)", preenchido(POLITICA_PUBLICA)),
    ("Governança (20%)", "Data do download", preenchido(r["data_download"])),
    ("Governança (20%)", "Filtros usados no ISP Microdados", preenchido(r["filtros"])),
    ("Governança (20%)", "Compromisso com o Termo de Responsabilidade", r["termo_de_responsabilidade"] is True),
    ("Governança (20%)", "Fonte usada foi o ISP Microdados (plano B aceito com justificativa)", fonte_usada == "ISP Microdados"),
    ("Governança (20%)", "Checagens de qualidade executadas", "qualidade_silver" in TABELAS),
    ("Governança (20%)", "Catálogo com dono em todas as tabelas", all(preenchido(d) for d in catalogo["dono"])),
]
resultado = pd.DataFrame(itens, columns=["criterio", "item", "ok"])
resultado["status"] = resultado["ok"].map({True: "OK", False: "FALTA"})
mostrar(resultado[["criterio", "item", "status"]], n=50)
faltam = int((~resultado["ok"]).sum())
print(f"\n{len(resultado) - faltam} de {len(resultado)} itens prontos.")
if fonte_usada != "ISP Microdados":
    print("Atenção: o grupo usou o plano B. Explique na apresentação por que o ISP Microdados não foi usado.")

print("\nRoteiro sugerido da apresentação (12 min):")
print(f"1. Pressão e decisão (1 min): {DECISAO['manchete_ou_pressao']} → {DECISAO['decisao_em_jogo']}")
print(f"2. Critérios e pesos (2 min): " + ", ".join(f"{c} {PESOS[c]:.0%}" for c in CRITERIOS) + f"; limiar de evidência {LIMIAR_EVIDENCIA_PCT}%")
print(f"3. Dados e governança (2 min): {fonte_usada}; download em {r['data_download']}; Termo: {'sim' if r['termo_de_responsabilidade'] else 'NÃO'}")
print(f"4. Evidência (3 min): CISP {CISP_FOCO}, {DELITO_FOCO}, variação {VARIACAO_FOCO}%; notícias: " + "; ".join(checagem_noticias["veredito"]))
print(f"5. Decisão e risco (2 min): {DECISAO_FINAL['decisao']} | risco: {DECISAO_FINAL['risco_de_errar']['o_que_pode_estar_errado']}")
print(f"6. Política pública (1,5 min): {POLITICA_PUBLICA['acao']} → {POLITICA_PUBLICA['indicador_de_acompanhamento']}")

## Depois da aula

- **Feedback de 8 min** à tarde: anotem cada ponto numa tabela *ponto do feedback → o que mudamos*. A versão final precisa responder a **todos** os pontos (condição para a nota).
- **Entrega até 27/11/2026**: este notebook exportado (*File → Export → IPython Notebook*), os slides e a tabela de feedback. **Não anexem a base do ISP Microdados**: ela fica só no workspace de quem baixou.

**Referências do lab:** ISP-RJ, [ISPDados](https://www.ispdados.rj.gov.br/) e [ISP Microdados](https://ispconecta.rj.gov.br/microdados/); Portaria ISP nº 127/2022; ALMEIDA, M. G. de; BOTEGA, L. C.; LIMA, P. M. de P. *Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese*, KDMiLe/SBC, 2026 (corpus CC BY 4.0); SAATY, T. L., *Int. J. Services Sciences*, 2008; DAMA-DMBOK2, cap. 13 (Qualidade de Dados).